In [ ]:
'''
K Nearest Neighbors
-----------------------
- Both Classification and Regression
- Classification -> KNeighborsClassifier
- Regression -> KNeighborsRegressor

- Classification
    - Calculate distance between data points to choose nearest neighbors to identify new data belongs to which class.

- Regression
    - Calculate distance between data points to get average data as a prediction value.

Finding optimal number of neighbors - Using Elbow Method Figure
-------------------------------------------------------------------
1. Error Rate
    - Maximum drop point of figure.
    
2. Accuracy Score
    - Highest Peak Point of figure.

Calculation of Elbow Method
-----------------------------
- Error Rate
    -> Number of wrong predictions

- Accuracy Score
    -> Accuracy of each execution of model.

k_range = range(1, 21) # 1 - 20
error_rate = []
acc_score = []

for i in k_range:
    knn = KNeighborsClassifier()
    knn.fit(X_train_scale, Y_train)
    knn_pred = knn.predict(X_test_scale)
    
    error_rate.append(np.mean(Y_test != knn_pred)) -> number of wrong prediction / total number of Y_test data
    acc_score.append(Y_test, knn_pred)

1000 Records -> population

Train -> 800
Test -> 200

emp_satis, emp_position, department    Knn_pred_label    Acc
1.5           A            CC              CC            0.85
3.5           B            ZZ              ZZ            0.43
4.6           D            yy              YY            0.75

Cross Validation
---------------------
- StratifiyFold
- cross_val_score

Overfitting and Underfitting
--------------------------------
Overfit -> when model learn more outliers data than important data
underfit -> when model learn all data but does not learn similar group data.

- Grid Search CV
- Random Search CV
'''

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Import KNN Model
# For KNN Regression -> KNeighborsRegressor
from sklearn.neighbors import KNeighborsClassifier
# Train Test Package
from sklearn.model_selection import train_test_split
# Data Preprocessing Package
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Model metrics Package
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [4]:
df = pd.read_csv('dataset/student.csv')
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,very good,medium,high,very low,very low,good,6,5,6,6
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,excellent,medium,medium,very low,very low,good,4,5,5,6
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,very good,medium,low,low,medium,good,10,7,8,10
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,good,low,low,very low,very low,excellent,2,15,14,15
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,very good,medium,low,very low,low,excellent,4,6,10,10


In [6]:
# Change grade to label
df['G3'].sort_values(ascending=False).unique()

array([20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10,  9,  8,  7,  6,  5,  4,
        0])

In [8]:
def convertGrade(grade):
    if grade >= 16:
        return 'A'
    elif grade >= 11:
        return 'B'
    elif grade >= 6:
        return 'C'
    else:
        return 'D'

In [9]:
df['Grade'] = df['G3'].apply(convertGrade)

In [13]:
df['Grade'].value_counts()

Grade
B    169
C    140
D     46
A     40
Name: count, dtype: int64

In [14]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'Grade'],
      dtype='object')

In [28]:
columns = df[['Medu', 'Fedu', 'traveltime', 'studytime', 'famsup', 'freetime', 
              'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'Grade']]

columns_to_encode = ['failures', 'schoolsup', 'internet', 'famrel']

# columns

# df['health'].unique()

## Label Encoding

In [26]:
'''
1. fit_transform() -> convert values to 0, 1, 2, 3, ..., n
2. inverse_transform() -> convert converted number back to label data.
    data = encoder.inverse_transform(df['failures_id'])
    data
'''
encoder = LabelEncoder()

for col in columns_to_encode:
    df[f'{col}_id'] = encoder.fit_transform(df[col])
print('Data converted...')

Data converted...


## Function to Manual Encode

In [29]:
def encodeData(dataframe, col_name, *args):
    encode_data = {}
    for idx, val in enumerate(args):
        encode_data[val] = idx

    dataframe[f'{col_name}_id'] = dataframe[col_name].map(encode_data)
    print(f'{col_name} data encoded...')

In [34]:
df['Fedu'].unique()

array(['higher education', 'primary education (4th grade)',
       '5th to 9th grade', 'secondary education', 'none'], dtype=object)

In [32]:
encodeData(df, 'Medu', 'none', 'primary education (4th grade)', '5th to 9th grade', 'secondary education', 'higher education')

Medu data encoded...


In [35]:
encodeData(df, 'Fedu', 'none', 'primary education (4th grade)', '5th to 9th grade', 'secondary education', 'higher education')

Fedu data encoded...


In [36]:
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,G1,G2,G3,Grade,failures_id,schoolsup_id,internet_id,famrel_id,Medu_id,Fedu_id
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,5,6,6,C,0,1,0,4,4,4
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,5,5,6,C,0,0,1,1,1,1
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,7,8,10,C,3,1,1,4,1,1
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,15,14,15,B,0,0,1,2,4,2
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,6,10,10,C,0,0,0,4,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,5th to 9th grade,5th to 9th grade,services,services,...,9,9,9,C,2,0,0,1,2,2
391,MS,M,17,U,LE3,T,secondary education,primary education (4th grade),services,services,...,14,16,16,A,0,0,1,0,3,1
392,MS,M,21,R,GT3,T,primary education (4th grade),primary education (4th grade),other,other,...,10,8,7,C,3,0,0,1,1,1
393,MS,M,18,R,LE3,T,secondary education,5th to 9th grade,services,other,...,11,12,10,C,0,0,1,4,3,2
